<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Blend_DermaMNIST_01_Attack_and_Feature_Save_Q1_PATCHED_1_5_10_COLAB_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blend / DermaMNIST 01 — Attack and Feature Save

Frozen Blend attack run. This notebook trains/loads only the poisoned QNN, verifies clean accuracy and ASR, then saves reusable feature tensors for Notebook 2. It does not run classical baseline, QXAI, sanitization, pair search, or detection.

In [1]:
# ============================================================
# Imports, configuration, reproducibility — Blend / DermaMNIST
# ============================================================

import os, sys, json, math, random, subprocess
from dataclasses import dataclass, asdict
from typing import Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA, PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

INSTALL_MISSING_PACKAGES = True

def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        if not INSTALL_MISSING_PACKAGES:
            raise
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
        return __import__(import_name)

medmnist = ensure_package("medmnist")
from medmnist import INFO
qml = ensure_package("pennylane")

ATTACK_NAME = "blend"
DATASET_NAME = "dermamnist"
RUN_DERMAMNIST_BLEND = True
RUN_PATHMNIST_FIBA = False
RUN_QTROJAN = False
RUN_CLASSICAL_POISONED_BASELINE = False
RUN_PAIR_SELECTION = False
RUN_QSENTRY_USEFULNESS_ABLATION = False
RUN_SUPERIORITY_MODULES = False
RUN_POISON_RATE_TRAINING_SWEEP = False
RUN_STEALTH_STRENGTH_SWEEP = False
RUN_QXAI_BLEND_EXPLANATION = False
RUN_BLEND_DATA_SANITIZATION = False

DEFAULT_OUT_DIR = "/kaggle/working/outputs_blend_dermamnist" if os.path.exists("/kaggle/working") else "/mnt/data/outputs_blend_dermamnist" if os.path.exists("/mnt/data") else "./outputs_blend_dermamnist"

CANDIDATE_PAIRS_BLEND = [(0, 4), (1, 4), (2, 4), (5, 4), (6, 4)]

@dataclass
class QMedShieldBlendConfig:
    primary_seed: int = 42
    dataset_name: str = DATASET_NAME
    attack_type: str = ATTACK_NAME
    n_classes: int = 7
    native_image_size: int = 28
    model_image_size: int = 96
    source_class: Optional[int] = None
    target_class: Optional[int] = None
    max_train_n: int = 5000
    pilot_max_train_n: int = 1500
    max_clean_test_n: int = 1200
    max_asr_n: int = 800
    val_size: float = 0.15
    use_imagenet_weights: bool = True
    freeze_resnet_lower_blocks: bool = True
    n_qubits: int = 8
    vqc_layers: int = 4
    obs_mode: str = "ZX_ALT"
    train_batch_size: int = 48
    eval_batch_size: int = 96
    clean_epochs: int = 8
    pilot_epochs: int = 2
    lr_base: float = 5e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.10
    clean_acc_min: float = 0.50
    asr_min: float = 0.80
    strict_attack_gate: bool = True
    ica_components: int = 4
    kmeans_n_init: int = 10
    fpr_target: float = 0.05
    k_values: Tuple[int, ...] = (3, 5, 8, 10)
    threshold_method: str = "clean_val_95pct"
    clean_source_n: int = 450
    clean_target_n: int = 500
    poison_n: int = 50
    blend_alpha: float = 0.20
    blend_alpha_values: Tuple[float, ...] = (0.20, 0.15, 0.10, 0.07, 0.05)
    blend_poison_rate: float = 0.15
    qsentry_poison_ratios: Tuple[float, ...] = (0.01, 0.05, 0.10)
    superiority_poison_rates: Tuple[float, ...] = (0.01, 0.05, 0.10)
    stealth_alpha_values: Tuple[float, ...] = (0.20, 0.15, 0.10, 0.07, 0.05)
    superiority_sweep_epochs: int = 8
    sanitization_epochs: int = 2
    # Active probe logic: evaluate how model measurements change when a suspected Blend trigger is re-applied.
    # This is not test tuning; alphas are fixed before evaluation and used for both validation and test pools.
    active_probe_alphas: Tuple[float, ...] = (0.03, 0.07, 0.15)
    qnn_ensemble_top_m_values: Tuple[int, ...] = (1, 2, 3, 5, 999)
    qsentry_threshold_methods: Tuple[str, ...] = ("top_expected_poison_count", "clean_val_95pct")
    out_dir: str = DEFAULT_OUT_DIR

cfg = QMedShieldBlendConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NEEDS_VERIFICATION = []
PLANNED_PENDING = []

assert ATTACK_NAME == "blend"
assert DATASET_NAME == "dermamnist"
assert RUN_DERMAMNIST_BLEND is True
assert RUN_PATHMNIST_FIBA is False
assert RUN_QTROJAN is False
assert cfg.attack_type == "blend" and cfg.dataset_name == "dermamnist"


def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(cfg.primary_seed)
print("Device:", DEVICE)
print("Output directory:", cfg.out_dir)
display(pd.DataFrame([asdict(cfg)]).T.rename(columns={0: "value"}))


# Colab-safe split controls
FREEZE_BLEND_CONFIG = True
RUN_PAIR_SEARCH = False
RUN_25_PILOT_SEARCH = False
RUN_CLASSICAL_BASELINE_IN_NOTEBOOK1 = False
RUN_QXAI_IN_NOTEBOOK1 = False
RUN_SANITIZATION = False
SAVE_FEATURE_TENSORS = True
LOAD_FEATURE_TENSORS_IF_AVAILABLE = True
SKIP_COMPLETED_STAGES = True
RUN_TRIGGER_TYPE_VALIDATION = False  # optional validation-only diagnostic, disabled by default

# Frozen validated Blend configuration. Do not rerun pair search unless explicitly enabled.
cfg.source_class = 5
cfg.target_class = 4
cfg.blend_alpha = 0.07
cfg.blend_poison_rate = 0.15
assert cfg.source_class == 5 and cfg.target_class == 4
assert abs(cfg.blend_alpha - 0.07) < 1e-12
assert abs(cfg.blend_poison_rate - 0.15) < 1e-12


# Colab ZIP export controls
# After Notebook 1 finishes, it zips outputs_blend_dermamnist/ and downloads it.
# Upload this ZIP into Notebook 2 when asked.
AUTO_ZIP_AND_DOWNLOAD_OUTPUTS = True
OUTPUT_ZIP_NAME = "outputs_blend_dermamnist.zip"
OUTPUT_ZIP_PATH = os.path.abspath(OUTPUT_ZIP_NAME)


Device: cpu
Output directory: ./outputs_blend_dermamnist


,value
primary_seed,42
dataset_name,dermamnist
attack_type,blend
n_classes,7
native_image_size,28
model_image_size,96
source_class,None
target_class,None
max_train_n,5000
pilot_max_train_n,1500


In [2]:
# ============================================================
# Dataset loading utilities
# ============================================================

def load_medmnist_split(dataset_name: str, split: str, download: bool = True):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    ds = DataClass(split=split, download=download)
    X = torch.tensor(ds.imgs, dtype=torch.float32)
    if X.ndim == 3:
        X = X.unsqueeze(-1)
    X = X.permute(0, 3, 1, 2) / 255.0
    if X.shape[1] == 1:
        X = X.repeat(1, 3, 1, 1)
    y = torch.tensor(ds.labels.squeeze(), dtype=torch.long)
    return X, y, info


def label_map_to_int(label_map):
    out = {}
    for k, v in label_map.items():
        name = v[0] if isinstance(v, (list, tuple)) else str(v)
        out[int(k)] = name
    return out


def load_dataset_bundle(dataset_name: str):
    X_train, y_train, info = load_medmnist_split(dataset_name, "train")
    X_val, y_val, _ = load_medmnist_split(dataset_name, "val")
    X_test, y_test, _ = load_medmnist_split(dataset_name, "test")
    class_map = label_map_to_int(info["label"])
    print(f"Loaded {dataset_name}: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")
    display(pd.DataFrame([{"class_id": k, "class_name": v} for k, v in class_map.items()]))
    return {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val, "X_test": X_test, "y_test": y_test, "class_map": class_map, "info": info}


def resize_tensor_images(X, size):
    if X.shape[-1] == size and X.shape[-2] == size:
        return X.float()
    return F.interpolate(X.float(), size=(size, size), mode="bilinear", align_corners=False)


def clone_cfg(base_cfg, **updates):
    data = asdict(base_cfg)
    data.update(updates)
    return type(base_cfg)(**data)


In [3]:
# ============================================================
# Model architecture and training helpers
# ============================================================

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)


def preprocess_for_resnet(x, model_image_size):
    x = F.interpolate(x, size=(model_image_size, model_image_size), mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def measurement_dim(n_qubits, obs_mode):
    obs = obs_mode.upper().strip()
    if obs in ["Z_ONLY", "ZX_ALT"]:
        return n_qubits
    if obs == "XYZ":
        return 3 * n_qubits
    raise ValueError(f"Unknown obs_mode={obs_mode}")


def make_vqc_qnode(n_qubits, n_layers, obs_mode="ZX_ALT"):
    try:
        dev = qml.device("lightning.qubit", wires=n_qubits)
    except Exception:
        print("[warning] lightning.qubit unavailable; using default.qubit")
        dev = qml.device("default.qubit", wires=n_qubits)
    obs = obs_mode.upper().strip()

    @qml.qnode(dev, interface="torch", diff_method="best")
    def circuit(angles, weights, input_scales, input_bias, trojan_angles, trigger_flag):
        for layer in range(n_layers):
            for q in range(n_qubits):
                qml.Rot(weights[layer, q, 0], weights[layer, q, 1], weights[layer, q, 2], wires=q)
            for q in range(n_qubits):
                qml.RY(input_scales[layer, q] * angles[q] + input_bias[layer, q], wires=q)
            # No QTrojan is used in this notebook; trigger_flag stays 0. This parameter remains only because the shared VQC layer supports QTrojan notebooks.
            for q in range(n_qubits):
                qml.RY(trigger_flag * trojan_angles[layer, q], wires=q)
            if layer % 2 == 0:
                for q in range(n_qubits):
                    qml.CNOT(wires=[q, (q + 1) % n_qubits])
            else:
                for q in range(n_qubits):
                    qml.CNOT(wires=[(q + 1) % n_qubits, q])
        if obs == "Z_ONLY":
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]
        if obs == "XYZ":
            out = []
            for q in range(n_qubits):
                out.extend([qml.expval(qml.PauliX(q)), qml.expval(qml.PauliY(q)), qml.expval(qml.PauliZ(q))])
            return out
        return [qml.expval(qml.PauliZ(q)) if q % 2 == 0 else qml.expval(qml.PauliX(q)) for q in range(n_qubits)]
    return circuit


class StrongVQCMeasurementLayer(nn.Module):
    def __init__(self, in_dim, n_qubits, n_layers, obs_mode, trojan_init_std=0.0):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.obs_mode = obs_mode
        self.compress = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh(),
        )
        self.weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))
        self.input_scales = nn.Parameter(torch.ones(n_layers, n_qubits))
        self.input_bias = nn.Parameter(torch.zeros(n_layers, n_qubits))
        self.trojan_angles = nn.Parameter(trojan_init_std * torch.randn(n_layers, n_qubits))
        self.qnode = make_vqc_qnode(n_qubits, n_layers, obs_mode)

    def forward(self, context, trojan_mask=None):
        angles = math.pi * (self.compress(context) + 1.0) / 2.0
        if trojan_mask is None:
            trojan_mask = torch.zeros(angles.shape[0], device=angles.device, dtype=angles.dtype)
        else:
            trojan_mask = trojan_mask.to(device=angles.device, dtype=angles.dtype).view(-1)
        vals = [
            torch.stack(self.qnode(a, self.weights, self.input_scales, self.input_bias, self.trojan_angles, flag)).float()
            for a, flag in zip(angles, trojan_mask)
        ]
        return torch.stack(vals, dim=0)


class QMedShieldHybridQNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        weights = None
        if cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception as exc:
            msg = f"needs verification: ImageNet weights unavailable ({exc}); using random ResNet-18 init."
            print("[warning]", msg)
            NEEDS_VERIFICATION.append(msg)
            resnet = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False
        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.vqc = StrongVQCMeasurementLayer(512, cfg.n_qubits, cfg.vqc_layers, cfg.obs_mode, 0.0)
        self.q_head = nn.Linear(measurement_dim(cfg.n_qubits, cfg.obs_mode), cfg.n_classes)

    def _context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.view(B, C, H*W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def forward(self, x_preprocessed, t=0.0, return_measurements=False):
        ctx = self._context(x_preprocessed)
        mask = torch.zeros(ctx.shape[0], device=ctx.device, dtype=ctx.dtype)
        qfeat = self.vqc(ctx, trojan_mask=mask)
        logits = self.q_head(qfeat)
        if return_measurements:
            return logits, qfeat, ctx
        return logits


class ClassicalResNetBiGRU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        weights = None
        if cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception:
            resnet = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False
        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Linear(512, cfg.n_classes)

    def _context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.view(B, C, H*W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def forward(self, x_preprocessed, return_context=False):
        ctx = self._context(x_preprocessed)
        logits = self.head(ctx)
        if return_context:
            return logits, ctx
        return logits


def verify_architecture(model, cfg, device):
    model = model.to(device)
    dummy = torch.rand(4, 3, cfg.native_image_size, cfg.native_image_size, device=device)
    xp = preprocess_for_resnet(dummy, cfg.model_image_size)
    logits, qfeat, ctx = model(xp, return_measurements=True)
    assert logits.shape == (4, cfg.n_classes), logits.shape
    assert qfeat.shape == (4, measurement_dim(cfg.n_qubits, cfg.obs_mode)), qfeat.shape
    assert ctx.shape == (4, 512), ctx.shape
    print("Architecture verified:", logits.shape, qfeat.shape, ctx.shape)


def stratified_limit_tensors(X, y, max_n, seed):
    if max_n is None or len(y) <= max_n:
        return X, y, np.arange(len(y))
    idx = np.arange(len(y))
    _, keep = train_test_split(idx, test_size=max_n, random_state=seed, stratify=y.cpu().numpy())
    keep = np.sort(keep)
    return X[keep], y[keep], keep


def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X.float(), y.long()), batch_size=batch_size, shuffle=shuffle, num_workers=0)


def optimizer_for_qnn(model, cfg):
    return torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": cfg.lr_base},
        {"params": model.vqc.parameters(), "lr": cfg.lr_base},
        {"params": model.q_head.parameters(), "lr": cfg.lr_base},
    ], weight_decay=cfg.weight_decay)


def optimizer_for_classical(model, cfg):
    return torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": cfg.lr_base},
        {"params": model.head.parameters(), "lr": cfg.lr_base},
    ], weight_decay=cfg.weight_decay)


@torch.no_grad()
def evaluate_classifier(model, X, y, cfg, device):
    model.eval()
    loader = make_loader(X, y, cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
        logits = model(xp)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1)); probs.append(prob)
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {"accuracy": float(accuracy_score(y_np, preds)), "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)), "preds": preds, "probs": probs}


@torch.no_grad()
def evaluate_classical_classifier(model, X, y, cfg, device):
    model.eval()
    loader = make_loader(X, y, cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
        logits = model(xp)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1)); probs.append(prob)
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {"accuracy": float(accuracy_score(y_np, preds)), "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)), "preds": preds, "probs": probs}


def train_clean_stage(model, train_loader, val_loader, cfg, device):
    model = model.to(device)
    opt = optimizer_for_qnn(model, cfg)
    hist = []
    print(f"QNN poisoned/clean training: {cfg.clean_epochs} epochs")
    for ep in range(1, cfg.clean_epochs + 1):
        model.train(); loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size); yb = yb.to(device)
            opt.zero_grad(); logits = model(xp); loss = F.cross_entropy(logits, yb, label_smoothing=cfg.label_smoothing)
            loss.backward(); opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb); n += len(xb)
        val = evaluate_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device)
        row = {"epoch": ep, "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]}
        hist.append(row)
        print(f"Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_acc']:.4f}")
    return model, hist


def train_classical_stage(model, train_loader, val_loader, cfg, device, label="classical_poisoned"):
    model = model.to(device)
    opt = optimizer_for_classical(model, cfg)
    hist = []
    print(f"[{label}] training: {cfg.clean_epochs} epochs")
    for ep in range(1, cfg.clean_epochs + 1):
        model.train(); loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size); yb = yb.to(device)
            opt.zero_grad(); logits = model(xp); loss = F.cross_entropy(logits, yb, label_smoothing=cfg.label_smoothing)
            loss.backward(); opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb); n += len(xb)
        val = evaluate_classical_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device)
        row = {"epoch": ep, "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]}
        hist.append(row)
        print(f"[{label}] Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_acc']:.4f}")
    return model, hist


In [4]:
# ============================================================
# Feature extraction, detector, ablation, plotting, and claim control
# ============================================================

@torch.no_grad()
def extract_qnn_features(model, X, cfg, device):
    model.eval(); q_rows, ctx_rows, prob_rows = [], [], []
    for s in range(0, len(X), cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+cfg.eval_batch_size].to(device), cfg.model_image_size)
        logits, q, ctx = model(xb, return_measurements=True)
        q_rows.append(q.detach().cpu().numpy())
        ctx_rows.append(ctx.detach().cpu().numpy())
        prob_rows.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return np.concatenate(q_rows), np.concatenate(ctx_rows), np.concatenate(prob_rows)


@torch.no_grad()
def extract_classical_context(model, X, cfg, device):
    if model is None:
        return None
    model.eval(); rows = []
    for s in range(0, len(X), cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+cfg.eval_batch_size].to(device), cfg.model_image_size)
        _, ctx = model(xb, return_context=True)
        rows.append(ctx.detach().cpu().numpy())
    return np.concatenate(rows)



def sanitize_feature_matrix(X, name="feature_matrix"):
    """Return a finite float32 2-D feature matrix and record any repair in NEEDS_VERIFICATION."""
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.ndim != 2:
        X = X.reshape(X.shape[0], -1)
    bad = ~np.isfinite(X)
    if bad.any():
        NEEDS_VERIFICATION.append(
            f"needs verification: sanitized {int(bad.sum())} non-finite values in {name}."
        )
        X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6).astype(np.float32)
    return X


def quantum_readout_statistics(q):
    """Finite readout statistics for expectation values q in [-1, 1]."""
    q = sanitize_feature_matrix(q, "quantum_readout_input").astype(np.float64)
    q = np.clip(q, -1.0 + 1e-6, 1.0 - 1e-6)
    eps = 1e-6
    p = np.clip((q + 1.0) / 2.0, eps, 1.0 - eps)
    entropy = -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))
    saturation = np.abs(q)
    uncertainty = 1.0 - saturation
    out = np.concatenate([q, entropy, saturation, uncertainty], axis=1)
    return sanitize_feature_matrix(out, "quantum_readout_statistics")


def block_standardize(train_block, test_block):
    scaler = StandardScaler()
    train_z = scaler.fit_transform(train_block)
    test_z = scaler.transform(test_block)
    scale = np.sqrt(max(train_z.shape[1], 1))
    return train_z / scale, test_z / scale


def make_block_weighted_hybrid(val_blocks, test_blocks, weights=None):
    if weights is None:
        weights = {k: 1.0 for k in val_blocks}
    val_parts, test_parts = [], []
    for name in val_blocks:
        v, t = block_standardize(val_blocks[name], test_blocks[name])
        val_parts.append(weights.get(name, 1.0) * v)
        test_parts.append(weights.get(name, 1.0) * t)
    return np.concatenate(val_parts, axis=1), np.concatenate(test_parts, axis=1)



def detector_scores_from_features(X_val_feat, y_val, X_test_feat, cfg, k):
    """
    Robust QSentry-style anomaly scoring.

    Fixes implemented:
    - sanitizes NaN/Inf before scaling/reduction;
    - treats FastICA non-convergence as a real failure and falls back to PCA;
    - refuses invalid K values instead of silently producing broken rows.
    """
    from sklearn.exceptions import ConvergenceWarning
    import warnings

    X_val_feat = sanitize_feature_matrix(X_val_feat, "detector_val_features")
    X_test_feat = sanitize_feature_matrix(X_test_feat, "detector_test_features")
    y_val = np.asarray(y_val).astype(int)
    k = int(k)
    if k < 2:
        raise ValueError(f"K must be >=2, got {k}")
    if len(X_val_feat) < k:
        raise ValueError(f"Validation feature count {len(X_val_feat)} is smaller than K={k}")

    scaler = StandardScaler()
    Vv0 = scaler.fit_transform(X_val_feat)
    Vt0 = scaler.transform(X_test_feat)
    Vv0 = sanitize_feature_matrix(Vv0, "scaled_val_features")
    Vt0 = sanitize_feature_matrix(Vt0, "scaled_test_features")

    n_comp = min(int(cfg.ica_components), Vv0.shape[1], max(1, Vv0.shape[0] - 1))
    try:
        reducer = FastICA(
            n_components=n_comp,
            random_state=int(cfg.primary_seed),
            whiten="unit-variance",
            max_iter=3000,
            tol=1e-4,
        )
        with warnings.catch_warnings():
            warnings.filterwarnings("error", category=ConvergenceWarning)
            Vv = reducer.fit_transform(Vv0)
            Vt = reducer.transform(Vt0)
    except Exception as exc:
        NEEDS_VERIFICATION.append(
            f"needs verification: FastICA fallback to PCA for K={k}: {exc}"
        )
        reducer = PCA(n_components=n_comp, random_state=int(cfg.primary_seed))
        Vv = reducer.fit_transform(Vv0)
        Vt = reducer.transform(Vt0)

    Vv = sanitize_feature_matrix(Vv, "reduced_val_features")
    Vt = sanitize_feature_matrix(Vt, "reduced_test_features")

    km = KMeans(n_clusters=k, random_state=int(cfg.primary_seed), n_init=int(cfg.kmeans_n_init))
    val_cluster = km.fit_predict(Vv)
    centers = km.cluster_centers_
    rows = []
    for c in range(k):
        mask = val_cluster == c
        rows.append((c, float(y_val[mask].mean()) if mask.any() else 0.0, int(mask.sum())))
    suspicious_cluster = sorted(rows, key=lambda x: (-x[1], x[2]))[0][0]

    def score(V):
        d_susp = np.linalg.norm(V - centers[suspicious_cluster], axis=1)
        other = [j for j in range(k) if j != suspicious_cluster]
        d_other = np.min(np.stack([np.linalg.norm(V - centers[j], axis=1) for j in other], axis=1), axis=1)
        return sanitize_feature_matrix((d_other - d_susp).reshape(-1, 1), "detector_scores").ravel()

    return score(Vv), score(Vt), {"k": k, "suspicious_cluster": int(suspicious_cluster), "cluster_rows": rows}


def prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count, method="clean_val_95pct", fpr=0.05):
    expected_poison_count = int(max(1, min(len(test_scores), round(expected_poison_count))))
    if method == "clean_val_95pct":
        clean_val_scores = val_scores[np.asarray(y_val) == 0]
        tau = float(np.percentile(clean_val_scores, 100 * (1 - fpr)))
        return (test_scores >= tau).astype(int), tau, "clean_val_95pct"
    if method not in ["top_expected_poison_count", "relative_cluster_size"]:
        raise ValueError(f"Unknown threshold method: {method}")
    pred = np.zeros(len(test_scores), dtype=int)
    top = np.argsort(test_scores)[-expected_poison_count:]
    pred[top] = 1
    return pred, None, method


def eval_detection(y_true, scores, pred):
    out = {
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "AUPRC": float(average_precision_score(y_true, scores)),
    }
    try:
        out["AUROC"] = float(roc_auc_score(y_true, scores))
    except Exception:
        out["AUROC"] = np.nan
    return out


def select_k_and_evaluate(feature_name, X_val_feat, y_val, X_test_feat, y_test, cfg, expected_poison_count, threshold_method=None):
    threshold_method = threshold_method or cfg.threshold_method
    val_rows, candidates = [], {}
    for k in cfg.k_values:
        try:
            val_scores, test_scores, meta = detector_scores_from_features(X_val_feat, y_val, X_test_feat, cfg, k)
            val_pred, val_tau, _ = prediction_from_scores(val_scores, y_val, val_scores, max(1, int(np.asarray(y_val).sum())), method=threshold_method, fpr=cfg.fpr_target)
            m_val = eval_detection(y_val, val_scores, val_pred)
            val_rows.append({"Feature Space": feature_name, "K": k, **m_val})
            candidates[k] = (val_scores, test_scores, meta)
        except Exception as e:
            val_rows.append({"Feature Space": feature_name, "K": k, "error": str(e)})
    val_df = pd.DataFrame(val_rows)
    valid = val_df.dropna(subset=["AUPRC", "F1"], how="any")
    if valid.empty:
        raise RuntimeError(f"No valid K for {feature_name}. Details: {val_df}")
    best_k = int(valid.sort_values(["AUPRC", "F1"], ascending=False).iloc[0]["K"])
    val_scores, test_scores, meta = candidates[best_k]
    test_pred, tau, threshold_name = prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count, method=threshold_method, fpr=cfg.fpr_target)
    test_metrics = eval_detection(y_test, test_scores, test_pred)
    row = {"Feature Space": feature_name, "K": best_k, "Threshold Method": threshold_name, "Tau": tau, **test_metrics}
    return row, val_df, test_scores


def plot_detection_score_distribution(scores, y_true, title, out_path):
    y_true = np.asarray(y_true)
    plt.figure(figsize=(7, 4))
    plt.hist(scores[y_true == 0], bins=40, alpha=0.65, label="clean")
    plt.hist(scores[y_true == 1], bins=40, alpha=0.65, label="poison")
    plt.title(title)
    plt.xlabel("Detection score")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()


def save_claim_decision(detection_df, attack_name, dataset_name, out_path):
    if detection_df.empty or "Feature Space" not in detection_df.columns:
        claim = "pending experiment"
        reason = "Detection table is empty or attack gate did not pass."
    else:
        def metric_for(name, metric):
            rows = detection_df[detection_df["Feature Space"] == name]
            if rows.empty:
                return np.nan
            return float(rows.iloc[0][metric])
        q_static_f1, q_static_auprc = metric_for("Quantum static measurement", "F1"), metric_for("Quantum static measurement", "AUPRC")
        q_delta_f1, q_delta_auprc = metric_for("Quantum probe-delta measurement", "F1"), metric_for("Quantum probe-delta measurement", "AUPRC")
        classical_f1, classical_auprc = metric_for("Classical context baseline", "F1"), metric_for("Classical context baseline", "AUPRC")
        qnnctx_f1, qnnctx_auprc = metric_for("QNN context baseline", "F1"), metric_for("QNN context baseline", "AUPRC")
        bal_f1, bal_auprc = metric_for("Balanced quantum+context hybrid", "F1"), metric_for("Balanced quantum+context hybrid", "AUPRC")
        context_f1 = np.nanmax([classical_f1, qnnctx_f1])
        context_auprc = np.nanmax([classical_auprc, qnnctx_auprc])
        quantum_best_f1 = np.nanmax([q_static_f1, q_delta_f1])
        quantum_best_auprc = np.nanmax([q_static_auprc, q_delta_auprc])
        if quantum_best_f1 > context_f1 and quantum_best_auprc > context_auprc:
            claim = "Strong"
            reason = "Quantum-only or quantum-delta features outperform classical/context features in both F1 and AUPRC."
        elif bal_f1 > context_f1 and bal_auprc > context_auprc:
            claim = "Moderate"
            reason = "Balanced quantum+context hybrid improves beyond classical/context alone."
        elif quantum_best_f1 > 0.30 or quantum_best_auprc > 0.30:
            claim = "Weak"
            reason = "Quantum features provide useful signal, but not primary detection power."
        else:
            claim = "Unsupported"
            reason = "Classical/context features carry most of the detection performance."
    text = f"""# Claim decision — {attack_name} / {dataset_name}\n\nFinal claim category: **{claim}**\n\nReason: {reason}\n\nSafe wording: Do not claim quantum-measurement dominance unless the feature ablation table proves it. Use the table-generated category above.\n"""
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(text)
    return claim, reason


def save_qsentry_usefulness_decision(qsentry_df, out_path):
    """
    Conservative, table-driven decision for whether the Blend branch shows QSentry-style usefulness.
    This does not force quantum superiority. It compares raw, classical-context, QNN-context,
    quantum-only, and hybrid features at each poison ratio under top-expected-poison thresholding.
    """
    if qsentry_df is None or qsentry_df.empty:
        text = "# QSentry-style Blend usefulness decision\n\nStatus: **pending experiment**\n\nReason: ablation table is empty.\n"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(text)
        return "pending experiment"

    lines = [
        "# QSentry-style Blend usefulness decision",
        "",
        "Rule: evaluate poison rates 1%, 5%, and 10% using validation-selected K and top-expected-poison thresholding.",
        "Do not claim QNN/quantum dominance unless the table shows it.",
        "",
    ]

    top_df = qsentry_df[qsentry_df["Threshold Method"] == "top_expected_poison_count"].copy()
    decisions = []

    for ratio in sorted(top_df["Target Poison Ratio"].dropna().unique()):
        sub = top_df[np.isclose(top_df["Target Poison Ratio"], ratio)].copy()
        if sub.empty:
            continue

        def metric(name, m):
            r = sub[sub["Feature Space"] == name]
            if r.empty or m not in r.columns:
                return np.nan
            return float(r.iloc[0][m])

        raw_f1 = metric("Raw pixel baseline", "F1")
        classical_f1 = metric("Classical context baseline", "F1")
        qnn_f1 = metric("QNN context baseline", "F1")
        qstatic_f1 = metric("Quantum static measurement", "F1")
        qdelta_f1 = metric("Quantum probe-delta measurement", "F1")
        hybrid_f1 = metric("Balanced quantum+context hybrid", "F1")

        classical_auprc = metric("Classical context baseline", "AUPRC")
        qnn_auprc = metric("QNN context baseline", "AUPRC")
        qstatic_auprc = metric("Quantum static measurement", "AUPRC")
        qdelta_auprc = metric("Quantum probe-delta measurement", "AUPRC")
        hybrid_auprc = metric("Balanced quantum+context hybrid", "AUPRC")

        quantum_best_f1 = np.nanmax([qstatic_f1, qdelta_f1])
        quantum_best_auprc = np.nanmax([qstatic_auprc, qdelta_auprc])
        context_best_f1 = np.nanmax([classical_f1, qnn_f1])
        context_best_auprc = np.nanmax([classical_auprc, qnn_auprc])

        raw_beaten = bool(np.nanmax([qnn_f1, qstatic_f1, qdelta_f1, hybrid_f1]) > raw_f1)
        classical_beaten_by_quantum = bool(quantum_best_f1 > classical_f1 and quantum_best_auprc >= classical_auprc)
        classical_beaten_by_hybrid = bool(hybrid_f1 > classical_f1 and hybrid_auprc >= classical_auprc)

        if classical_beaten_by_quantum:
            verdict = "strong quantum-only advantage"
        elif classical_beaten_by_hybrid:
            verdict = "moderate hybrid advantage"
        elif raw_beaten:
            verdict = "weak useful signal over raw baseline"
        else:
            verdict = "unsupported at this ratio"

        decisions.append(verdict)
        lines += [
            f"## Target poison ratio: {ratio:.0%}",
            f"- Raw F1: `{raw_f1:.4f}`",
            f"- Classical context F1/AUPRC: `{classical_f1:.4f}` / `{classical_auprc:.4f}`",
            f"- QNN context F1/AUPRC: `{qnn_f1:.4f}` / `{qnn_auprc:.4f}`",
            f"- Best quantum-only F1/AUPRC: `{quantum_best_f1:.4f}` / `{quantum_best_auprc:.4f}`",
            f"- Balanced hybrid F1/AUPRC: `{hybrid_f1:.4f}` / `{hybrid_auprc:.4f}`",
            f"- Verdict: **{verdict}**",
            "",
        ]

    if any("strong" in d for d in decisions):
        overall = "Strong"
        safe = "Quantum-only features outperform classical context for at least one poison-rate setting."
    elif any("moderate" in d for d in decisions):
        overall = "Moderate"
        safe = "Hybrid quantum+context features outperform classical context for at least one poison-rate setting."
    elif any("weak" in d for d in decisions):
        overall = "Weak"
        safe = "QNN/quantum features improve over raw pixel clustering but do not consistently beat classical context."
    else:
        overall = "Unsupported"
        safe = "The ablation does not show a reliable QNN/quantum advantage."

    lines += [
        "## Overall decision",
        f"Final category: **{overall}**",
        "",
        f"Safe paper wording: {safe}",
        "",
        "If the category is Weak or Unsupported, do not write that QNN/quantum detection is superior to classical context.",
    ]

    text = "\n".join(lines) + "\n"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(text)
    return overall


In [5]:
# ============================================================
# Blend-specific attack, ASR, pools, probes, and runner
# ============================================================

def make_blend_trigger(shape, seed):
    rng = np.random.default_rng(seed)
    C, H, W = shape
    return torch.tensor(rng.random((1, C, H, W)), dtype=torch.float32)


def apply_blend_trigger(X, trigger, alpha):
    """
    Blend-style backdoor trigger.

    X: [N, C, H, W], values in [0, 1]
    trigger: [1, C, H, W] or [C, H, W], values in [0, 1]

    Preserves device/dtype, resizes trigger if needed, broadcasts batch dimension,
    and clips the result to [0, 1].
    """
    device = X.device
    dtype = X.dtype

    if trigger.ndim == 3:
        trigger = trigger.unsqueeze(0)

    trigger = trigger.to(device=device, dtype=dtype)

    if trigger.shape[-2:] != X.shape[-2:]:
        trigger = F.interpolate(
            trigger,
            size=X.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

    if trigger.shape[0] == 1 and X.shape[0] != 1:
        trigger = trigger.repeat(X.shape[0], 1, 1, 1)

    return ((1.0 - alpha) * X + alpha * trigger).clamp(0, 1)


def build_poisoned_blend_trainset(X_train_96, y_train, local_cfg, seed):
    y_np = y_train.cpu().numpy()
    src_idx = np.where(y_np == local_cfg.source_class)[0]
    if len(src_idx) == 0:
        raise ValueError("No source-class samples available for Blend poisoning.")
    rng = np.random.default_rng(seed)
    n_poison = max(1, int(round(len(src_idx) * local_cfg.blend_poison_rate)))
    n_poison = min(n_poison, len(src_idx))
    poison_idx = np.sort(rng.choice(src_idx, size=n_poison, replace=False))
    Xp, yp = X_train_96.clone(), y_train.clone()
    trigger = make_blend_trigger(tuple(Xp.shape[1:]), seed)
    Xp[poison_idx] = apply_blend_trigger(Xp[poison_idx], trigger, local_cfg.blend_alpha)
    yp[poison_idx] = local_cfg.target_class
    mask = torch.zeros(len(Xp), dtype=torch.bool); mask[poison_idx] = True
    return Xp, yp, mask, trigger


def benign_augment_for_blend(X):
    X_flip = torch.flip(X, dims=[3])
    X_noise = (X + 0.015 * torch.randn_like(X)).clamp(0, 1)
    return (X_flip + X_noise) / 2.0


def make_split_from_bundle(bundle, local_cfg, seed, max_train_n=None):
    # Critical runtime fix: limit at native 28x28 first, then resize only the selected subset.
    X_lim_native, y_lim, _ = stratified_limit_tensors(bundle["X_train"], bundle["y_train"], max_train_n or local_cfg.max_train_n, seed)
    X_lim = resize_tensor_images(X_lim_native, local_cfg.model_image_size)
    idx = np.arange(len(y_lim))
    tr, va = train_test_split(idx, test_size=local_cfg.val_size, random_state=seed, stratify=y_lim.cpu().numpy())
    return {"X_train": X_lim[np.sort(tr)], "y_train": y_lim[np.sort(tr)], "X_val": X_lim[np.sort(va)], "y_val": y_lim[np.sort(va)]}


@torch.no_grad()
def evaluate_blend_asr(model, X_raw, y, trigger, local_cfg, device, classical=False, seed=42):
    idx = np.where(y.cpu().numpy() == local_cfg.source_class)[0]
    if len(idx) == 0:
        raise ValueError("No source-class samples for Blend ASR.")
    rng = np.random.default_rng(seed)
    if local_cfg.max_asr_n and len(idx) > local_cfg.max_asr_n:
        idx = rng.choice(idx, local_cfg.max_asr_n, replace=False)
    idx = np.sort(idx)
    X_src = resize_tensor_images(X_raw[idx], local_cfg.model_image_size)
    X_trig = apply_blend_trigger(X_src, trigger, local_cfg.blend_alpha)
    y_target = torch.full((len(idx),), local_cfg.target_class, dtype=torch.long)
    if classical:
        return evaluate_classical_classifier(model, X_trig, y_target, local_cfg, device)
    return evaluate_classifier(model, X_trig, y_target, local_cfg, device)


def build_blend_detection_pool(X_raw, y, trigger, local_cfg, seed, poison_ratio=0.05, max_poison_n=None):
    """
    Build a QSentry-style minority poison pool.

    poison_ratio is the target poison fraction in the final pool:
        poison / (clean_source + clean_target + poison)

    The function uses only the provided split, never test data for validation pools.
    It preserves a clean-source + clean-target + triggered-source minority structure.
    """
    if not (0 < float(poison_ratio) < 0.5):
        raise ValueError(f"poison_ratio must be in (0, 0.5), got {poison_ratio}")

    y_np = y.cpu().numpy()
    src_idx = np.where(y_np == local_cfg.source_class)[0]
    tgt_idx = np.where(y_np == local_cfg.target_class)[0]
    rng = np.random.default_rng(seed)
    src_idx = rng.permutation(src_idx)
    tgt_idx = rng.permutation(tgt_idx)

    if len(src_idx) < 12 or len(tgt_idx) < 10:
        raise ValueError(
            f"Insufficient samples for Blend detection pool: "
            f"source={len(src_idx)}, target={len(tgt_idx)}"
        )

    clean_target_n = min(local_cfg.clean_target_n, len(tgt_idx))
    clean_source_n = min(local_cfg.clean_source_n, len(src_idx) - 1)

    # Desired poison count for the requested final poison fraction.
    base_clean_n = clean_source_n + clean_target_n
    desired_poison_n = max(1, int(round((float(poison_ratio) / (1.0 - float(poison_ratio))) * base_clean_n)))

    if max_poison_n is not None:
        desired_poison_n = min(desired_poison_n, int(max_poison_n))

    # If source samples are limited, reserve enough source images for poison while keeping at least 10 clean source.
    if clean_source_n + desired_poison_n > len(src_idx):
        clean_source_n = max(10, len(src_idx) - desired_poison_n)
        base_clean_n = clean_source_n + clean_target_n
        desired_poison_n = max(1, int(round((float(poison_ratio) / (1.0 - float(poison_ratio))) * base_clean_n)))

    poison_n = min(desired_poison_n, max(1, len(src_idx) - clean_source_n))

    if clean_source_n < 10 or clean_target_n < 10 or poison_n < 1:
        raise ValueError(
            f"Insufficient samples after ratio adjustment: "
            f"clean_source={clean_source_n}, clean_target={clean_target_n}, poison={poison_n}"
        )

    clean_src_idx = src_idx[:clean_source_n]
    poison_src_idx = src_idx[clean_source_n:clean_source_n + poison_n]
    clean_tgt_idx = tgt_idx[:clean_target_n]

    X_clean_src = resize_tensor_images(X_raw[clean_src_idx], local_cfg.model_image_size)
    X_clean_tgt = resize_tensor_images(X_raw[clean_tgt_idx], local_cfg.model_image_size)
    X_poison_src = resize_tensor_images(X_raw[poison_src_idx], local_cfg.model_image_size)
    X_poison = apply_blend_trigger(X_poison_src, trigger, local_cfg.blend_alpha)

    X_pool = torch.cat([X_clean_src, X_clean_tgt, X_poison], dim=0)
    y_poison = np.array([0] * (len(X_clean_src) + len(X_clean_tgt)) + [1] * len(X_poison), dtype=int)

    group = np.array(
        ["clean_source"] * len(X_clean_src)
        + ["clean_target"] * len(X_clean_tgt)
        + ["poison_triggered_source"] * len(X_poison)
    )

    actual_ratio = float(y_poison.mean())
    summary = pd.DataFrame({
        "Group": ["Clean source", "Clean target", "Poisonous triggered source", "Total", "Poison ratio", "Target poison ratio"],
        "Count": [len(X_clean_src), len(X_clean_tgt), len(X_poison), len(X_pool), f"{100 * actual_ratio:.2f}%", f"{100 * float(poison_ratio):.2f}%"],
    })
    display(summary)

    return X_pool, y_poison, group, summary

def extract_quantum_probe_features(qnn_model, X_pool, local_cfg, device):
    q0, _, probs0 = extract_qnn_features(qnn_model, X_pool, local_cfg, device)
    X_probe = benign_augment_for_blend(X_pool)
    q_probe, _, probs_probe = extract_qnn_features(qnn_model, X_probe, local_cfg, device)
    signed_delta = q_probe - q0
    abs_delta = np.abs(signed_delta)
    norm_delta = np.linalg.norm(signed_delta, axis=1, keepdims=True)
    target_conf = probs0[:, local_cfg.target_class:local_cfg.target_class+1]
    target_conf_delta = probs0[:, local_cfg.target_class:local_cfg.target_class+1] - probs_probe[:, local_cfg.target_class:local_cfg.target_class+1]
    return np.concatenate([q0, q_probe, signed_delta, abs_delta, norm_delta, target_conf, target_conf_delta], axis=1)


def extract_qmrs_full_hybrid_features(qnn_model, X_pool, local_cfg, device):
    q_probe = extract_quantum_probe_features(qnn_model, X_pool, local_cfg, device)
    _, ctx, _ = extract_qnn_features(qnn_model, X_pool, local_cfg, device)
    return np.concatenate([q_probe, ctx], axis=1)


def train_poisoned_model(bundle, local_cfg, seed, max_train_n=None, epochs_override=None, eval_split="val", train_classical_baseline=False):
    assert eval_split in ["val", "test"]
    split = make_split_from_bundle(bundle, local_cfg, seed, max_train_n=max_train_n)
    if epochs_override is not None:
        local_cfg = clone_cfg(local_cfg, clean_epochs=epochs_override)
    Xp, yp, poison_mask, trigger = build_poisoned_blend_trainset(split["X_train"], split["y_train"], local_cfg, seed)
    train_loader = make_loader(Xp, yp, local_cfg.train_batch_size, shuffle=True)
    val_loader = make_loader(split["X_val"], split["y_val"], local_cfg.eval_batch_size, shuffle=False)
    qnn = QMedShieldHybridQNN(local_cfg).to(DEVICE)
    verify_architecture(qnn, local_cfg, DEVICE)
    qnn, qnn_hist = train_clean_stage(qnn, train_loader, val_loader, local_cfg, DEVICE)
    classical, classical_hist = None, []
    if train_classical_baseline:
        classical = ClassicalResNetBiGRU(local_cfg).to(DEVICE)
        classical, classical_hist = train_classical_stage(classical, train_loader, val_loader, local_cfg, DEVICE, label="classical_blend_poisoned_final")
    if eval_split == "val":
        X_eval_raw, y_eval = split["X_val"], split["y_val"]
    else:
        X_eval_raw, y_eval = bundle["X_test"], bundle["y_test"]
    X_eval_lim_native, y_eval_lim, _ = stratified_limit_tensors(X_eval_raw, y_eval, local_cfg.max_clean_test_n, seed)
    X_eval_lim = resize_tensor_images(X_eval_lim_native, local_cfg.model_image_size)
    qnn_ca = evaluate_classifier(qnn, X_eval_lim, y_eval_lim, local_cfg, DEVICE)
    qnn_asr = evaluate_blend_asr(qnn, X_eval_raw, y_eval, trigger, local_cfg, DEVICE, classical=False, seed=seed)
    classical_ca = classical_asr = None
    if classical is not None:
        classical_ca = evaluate_classical_classifier(classical, X_eval_lim, y_eval_lim, local_cfg, DEVICE)
        classical_asr = evaluate_blend_asr(classical, X_eval_raw, y_eval, trigger, local_cfg, DEVICE, classical=True, seed=seed)
    return {"qnn": qnn, "classical": classical, "trigger": trigger, "poison_mask": poison_mask, "split": split, "qnn_ca": qnn_ca, "qnn_asr": qnn_asr, "classical_ca": classical_ca, "classical_asr": classical_asr, "local_cfg": local_cfg, "eval_split": eval_split, "classical_trained": bool(classical is not None)}


def pilot_validation_auprc(bundle, art, local_cfg, seed):
    # Small validation-only pool for pilot selection. Keeps pair selection practical and avoids using test data.
    pilot_cfg = clone_cfg(local_cfg, clean_source_n=80, clean_target_n=80, poison_n=10)
    X_val_pool, y_val_poison, _, _ = build_blend_detection_pool(bundle["X_val"], bundle["y_val"], art["trigger"], pilot_cfg, seed)
    qprobe_val = extract_quantum_probe_features(art["qnn"], X_val_pool, pilot_cfg, DEVICE)
    scores, _, _ = detector_scores_from_features(qprobe_val, y_val_poison, qprobe_val, pilot_cfg, k=pilot_cfg.k_values[0])
    return float(average_precision_score(y_val_poison, scores))


In [6]:


# ============================================================
# Optional trigger-type builders for validation-only diagnostics
# Default production run uses the frozen random Blend trigger for reproducibility.
# ============================================================
TRIGGER_TYPES = [
    "target_mean",
    "smoothed_target_mean",
    "low_contrast_pattern",
    "high_blur_target_mean",
    "channel_balanced_low_contrast",
    "augmentation_stable_low_contrast",
]

def make_blend_trigger_by_type(X_train_96, y_train, local_cfg, seed, trigger_type="frozen_random"):
    """Validation-only trigger construction helper. Never uses test data."""
    trigger_type = str(trigger_type).lower().strip()
    if trigger_type == "frozen_random":
        return make_blend_trigger(tuple(X_train_96.shape[1:]), seed)
    target_idx = torch.where(y_train == int(local_cfg.target_class))[0]
    if len(target_idx) == 0:
        raise ValueError(f"No target-class samples for trigger_type={trigger_type}")
    X_t = X_train_96[target_idx]
    if trigger_type in ["target_mean", "smoothed_target_mean", "high_blur_target_mean"]:
        trig = X_t.mean(dim=0, keepdim=True).clamp(0, 1)
        if trigger_type == "smoothed_target_mean":
            trig = F.avg_pool2d(trig, kernel_size=5, stride=1, padding=2).clamp(0, 1)
        if trigger_type == "high_blur_target_mean":
            trig = F.avg_pool2d(trig, kernel_size=11, stride=1, padding=5).clamp(0, 1)
        return trig
    if trigger_type == "low_contrast_pattern":
        rng = np.random.default_rng(seed)
        noise = torch.tensor(rng.normal(loc=0.5, scale=0.05, size=(1, X_train_96.shape[1], X_train_96.shape[2], X_train_96.shape[3])), dtype=X_train_96.dtype)
        return noise.clamp(0, 1)
    if trigger_type == "channel_balanced_low_contrast":
        trig = X_t.mean(dim=0, keepdim=True)
        flat = trig.view(1, trig.shape[1], -1)
        channel_mean = flat.mean(dim=-1).view(1, trig.shape[1], 1, 1)
        trig = (0.5 + 0.20 * (trig - channel_mean)).clamp(0, 1)
        return trig
    if trigger_type == "augmentation_stable_low_contrast":
        # Validation-only candidate: low-contrast pattern designed to be less raw-pixel obvious
        # while still perturbing the QNN measurement trajectory under active probes.
        trig = X_t.mean(dim=0, keepdim=True)
        trig = F.avg_pool2d(trig, kernel_size=9, stride=1, padding=4)
        flat = trig.view(1, trig.shape[1], -1)
        channel_mean = flat.mean(dim=-1).view(1, trig.shape[1], 1, 1)
        trig = (0.5 + 0.12 * (trig - channel_mean)).clamp(0, 1)
        return trig
    raise ValueError(f"Unknown trigger_type={trigger_type}")

def build_poisoned_blend_trainset_with_trigger(X_train_96, y_train, local_cfg, seed, trigger):
    """Build the poisoned trainset using a supplied trigger object, preserving train/test trigger identity."""
    y_np = y_train.cpu().numpy()
    src_idx = np.where(y_np == local_cfg.source_class)[0]
    if len(src_idx) == 0:
        raise ValueError("No source-class samples available for Blend poisoning.")
    rng = np.random.default_rng(seed)
    n_poison = max(1, int(round(len(src_idx) * local_cfg.blend_poison_rate)))
    n_poison = min(n_poison, len(src_idx))
    poison_idx = np.sort(rng.choice(src_idx, size=n_poison, replace=False))
    Xp, yp = X_train_96.clone(), y_train.clone()
    Xp[poison_idx] = apply_blend_trigger(Xp[poison_idx], trigger, local_cfg.blend_alpha)
    yp[poison_idx] = local_cfg.target_class
    mask = torch.zeros(len(Xp), dtype=torch.bool); mask[poison_idx] = True
    return Xp, yp, mask


In [7]:

# ============================================================
# Optional validation-only trigger-type diagnostic
# Disabled by default to prevent Colab timeout.
# ============================================================

def run_validation_only_trigger_type_test(bundle, base_cfg, seed):
    """
    Optional diagnostic. Uses validation only and never touches test data.
    This is for cases where the frozen trigger is too easy for classical baselines.
    Default RUN_TRIGGER_TYPE_VALIDATION=False, so the normal run does not spend time here.
    """
    if not RUN_TRIGGER_TYPE_VALIDATION:
        print("Trigger-type validation is disabled. Set RUN_TRIGGER_TYPE_VALIDATION=True only if strict QNN superiority fails.")
        return pd.DataFrame()
    rows = []
    split = make_split_from_bundle(bundle, base_cfg, seed, max_train_n=base_cfg.pilot_max_train_n)
    for trig_type in TRIGGER_TYPES:
        local_cfg = clone_cfg(base_cfg, clean_epochs=base_cfg.pilot_epochs)
        try:
            trigger = make_blend_trigger_by_type(split["X_train"], split["y_train"], local_cfg, seed, trig_type)
            Xp, yp, _ = build_poisoned_blend_trainset_with_trigger(split["X_train"], split["y_train"], local_cfg, seed, trigger)
            train_loader = make_loader(Xp, yp, local_cfg.train_batch_size, shuffle=True)
            val_loader = make_loader(split["X_val"], split["y_val"], local_cfg.eval_batch_size, shuffle=False)
            qnn = QMedShieldHybridQNN(local_cfg).to(DEVICE)
            verify_architecture(qnn, local_cfg, DEVICE)
            qnn, _ = train_clean_stage(qnn, train_loader, val_loader, local_cfg, DEVICE)
            val_ca = evaluate_classifier(qnn, split["X_val"], split["y_val"], local_cfg, DEVICE)
            val_asr = evaluate_blend_asr(qnn, split["X_val"], split["y_val"], trigger, local_cfg, DEVICE, classical=False, seed=seed)
            rows.append({
                "Trigger Type": trig_type,
                "Eval Split": "val",
                "Val CA": float(val_ca["accuracy"]),
                "Val ASR": float(val_asr["accuracy"]),
                "Gate Passed?": bool(float(val_asr["accuracy"]) >= local_cfg.asr_min and float(val_ca["accuracy"]) >= local_cfg.clean_acc_min),
                "Status": "ok",
            })
        except Exception as exc:
            rows.append({"Trigger Type": trig_type, "Eval Split": "val", "Status": "error", "error": str(exc), "Gate Passed?": False})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(base_cfg.out_dir, "blend_trigger_type_validation.csv"), index=False)
    display(df)
    return df


In [8]:

# ============================================================
# Notebook 1 main: frozen attack run + reusable feature saving
# ============================================================
FEATURE_FILES = {
    "qnn_context_val": "features_qnn_context_val.pt",
    "qnn_context_test": "features_qnn_context_test.pt",
    "quantum_static_val": "features_quantum_static_val.pt",
    "quantum_static_test": "features_quantum_static_test.pt",
    "quantum_probe_delta_val": "features_quantum_probe_delta_val.pt",
    "quantum_probe_delta_test": "features_quantum_probe_delta_test.pt",
    "quantum_readout_stats_val": "features_quantum_readout_stats_val.pt",
    "quantum_readout_stats_test": "features_quantum_readout_stats_test.pt",
    "raw_pixels_val": "features_raw_pixels_val.pt",
    "raw_pixels_test": "features_raw_pixels_test.pt",
    "labels_val": "labels_val.pt",
    "labels_test": "labels_test.pt",
}

CHECKPOINT_PATH = os.path.join(cfg.out_dir, "blend_qnn_poisoned_checkpoint.pt")
CONFIG_PATH = os.path.join(cfg.out_dir, "blend_config_used.json")
ATTACK_SUCCESS_PATH = os.path.join(cfg.out_dir, "blend_attack_success.csv")


def _all_notebook1_outputs_exist():
    paths = [CHECKPOINT_PATH, CONFIG_PATH, ATTACK_SUCCESS_PATH,
             os.path.join(cfg.out_dir, "pool_metadata_val.csv"),
             os.path.join(cfg.out_dir, "pool_metadata_test.csv")]
    paths += [os.path.join(cfg.out_dir, fn) for fn in FEATURE_FILES.values()]
    return all(os.path.exists(p) for p in paths)


def save_tensor_feature(name, value):
    path = os.path.join(cfg.out_dir, FEATURE_FILES[name])
    if isinstance(value, np.ndarray):
        value = torch.tensor(value, dtype=torch.float32)
    elif isinstance(value, torch.Tensor):
        value = value.detach().cpu()
    else:
        value = torch.tensor(np.asarray(value), dtype=torch.float32)
    torch.save(value, path)
    return path


def save_pool_metadata(path, group, labels):
    df = pd.DataFrame({
        "row_id": np.arange(len(labels), dtype=int),
        "group": np.asarray(group).astype(str),
        "poison_label": np.asarray(labels).astype(int),
    })
    df.to_csv(path, index=False)
    return df


def build_and_save_pool_features(qnn, bundle, trigger, local_cfg, seed):
    print("Building fixed 5% validation/test detection pools and saving reusable features...")
    X_val_pool, y_val_poison, group_val, _ = build_blend_detection_pool(
        bundle["X_val"], bundle["y_val"], trigger, local_cfg, seed + 101, poison_ratio=0.05
    )
    X_test_pool, y_test_poison, group_test, _ = build_blend_detection_pool(
        bundle["X_test"], bundle["y_test"], trigger, local_cfg, seed + 202, poison_ratio=0.05
    )

    q_val, qctx_val, _ = extract_qnn_features(qnn, X_val_pool, local_cfg, DEVICE)
    q_test, qctx_test, _ = extract_qnn_features(qnn, X_test_pool, local_cfg, DEVICE)
    qprobe_val = extract_quantum_probe_features(qnn, X_val_pool, local_cfg, DEVICE)
    qprobe_test = extract_quantum_probe_features(qnn, X_test_pool, local_cfg, DEVICE)
    qstats_val = quantum_readout_statistics(q_val)
    qstats_test = quantum_readout_statistics(q_test)
    raw_val = X_val_pool.reshape(len(X_val_pool), -1).numpy().astype(np.float32)
    raw_test = X_test_pool.reshape(len(X_test_pool), -1).numpy().astype(np.float32)

    saved = {}
    saved["features_qnn_context_val.pt"] = save_tensor_feature("qnn_context_val", qctx_val)
    saved["features_qnn_context_test.pt"] = save_tensor_feature("qnn_context_test", qctx_test)
    saved["features_quantum_static_val.pt"] = save_tensor_feature("quantum_static_val", q_val)
    saved["features_quantum_static_test.pt"] = save_tensor_feature("quantum_static_test", q_test)
    saved["features_quantum_probe_delta_val.pt"] = save_tensor_feature("quantum_probe_delta_val", qprobe_val)
    saved["features_quantum_probe_delta_test.pt"] = save_tensor_feature("quantum_probe_delta_test", qprobe_test)
    saved["features_quantum_readout_stats_val.pt"] = save_tensor_feature("quantum_readout_stats_val", qstats_val)
    saved["features_quantum_readout_stats_test.pt"] = save_tensor_feature("quantum_readout_stats_test", qstats_test)
    saved["features_raw_pixels_val.pt"] = save_tensor_feature("raw_pixels_val", raw_val)
    saved["features_raw_pixels_test.pt"] = save_tensor_feature("raw_pixels_test", raw_test)
    torch.save(torch.tensor(y_val_poison, dtype=torch.long), os.path.join(local_cfg.out_dir, FEATURE_FILES["labels_val"]))
    torch.save(torch.tensor(y_test_poison, dtype=torch.long), os.path.join(local_cfg.out_dir, FEATURE_FILES["labels_test"]))

    save_pool_metadata(os.path.join(local_cfg.out_dir, "pool_metadata_val.csv"), group_val, y_val_poison)
    save_pool_metadata(os.path.join(local_cfg.out_dir, "pool_metadata_test.csv"), group_test, y_test_poison)
    return saved


def run_notebook1():
    os.makedirs(cfg.out_dir, exist_ok=True)
    assert FREEZE_BLEND_CONFIG is True
    assert RUN_PAIR_SEARCH is False and RUN_25_PILOT_SEARCH is False
    assert RUN_CLASSICAL_BASELINE_IN_NOTEBOOK1 is False
    assert RUN_QXAI_IN_NOTEBOOK1 is False
    assert ATTACK_NAME == "blend" and DATASET_NAME == "dermamnist"

    if SKIP_COMPLETED_STAGES and LOAD_FEATURE_TENSORS_IF_AVAILABLE and _all_notebook1_outputs_exist():
        print("All Notebook 1 outputs already exist. Skipping completed stages.")
        return pd.read_csv(ATTACK_SUCCESS_PATH)

    bundle = load_dataset_bundle(cfg.dataset_name)
    split = make_split_from_bundle(bundle, cfg, cfg.primary_seed, max_train_n=cfg.max_train_n)
    Xp, yp, poison_mask, trigger = build_poisoned_blend_trainset(split["X_train"], split["y_train"], cfg, cfg.primary_seed)

    train_loader = make_loader(Xp, yp, cfg.train_batch_size, shuffle=True)
    val_loader = make_loader(split["X_val"], split["y_val"], cfg.eval_batch_size, shuffle=False)

    qnn = QMedShieldHybridQNN(cfg).to(DEVICE)
    verify_architecture(qnn, cfg, DEVICE)
    qnn, hist = train_clean_stage(qnn, train_loader, val_loader, cfg, DEVICE)

    X_eval_raw, y_eval = bundle["X_test"], bundle["y_test"]
    X_eval_lim_native, y_eval_lim, _ = stratified_limit_tensors(X_eval_raw, y_eval, cfg.max_clean_test_n, cfg.primary_seed)
    X_eval_lim = resize_tensor_images(X_eval_lim_native, cfg.model_image_size)
    qnn_ca = evaluate_classifier(qnn, X_eval_lim, y_eval_lim, cfg, DEVICE)
    qnn_asr = evaluate_blend_asr(qnn, X_eval_raw, y_eval, trigger, cfg, DEVICE, classical=False, seed=cfg.primary_seed)

    row = {
        "Attack": ATTACK_NAME,
        "Dataset": DATASET_NAME,
        "Source": int(cfg.source_class),
        "Target": int(cfg.target_class),
        "Source→Target": f"{cfg.source_class}->{cfg.target_class}",
        "Strength/Alpha": float(cfg.blend_alpha),
        "Poison Rate": float(cfg.blend_poison_rate),
        "CA": float(qnn_ca["accuracy"]),
        "ASR": float(qnn_asr["accuracy"]),
        "Gate Passed?": bool(float(qnn_asr["accuracy"]) >= cfg.asr_min and float(qnn_ca["accuracy"]) >= cfg.clean_acc_min),
        "Status": "passed_attack_gate" if float(qnn_asr["accuracy"]) >= cfg.asr_min else "failed_attack_gate",
    }
    attack_df = pd.DataFrame([row])
    display(attack_df)
    attack_df.to_csv(ATTACK_SUCCESS_PATH, index=False)

    with open(CONFIG_PATH, "w") as f:
        json.dump({**asdict(cfg), "source_class": int(cfg.source_class), "target_class": int(cfg.target_class)}, f, indent=2)

    torch.save({
        "model_state_dict": qnn.state_dict(),
        "cfg": asdict(cfg),
        "trigger": trigger.detach().cpu(),
        "history": hist,
        "attack_success": row,
    }, CHECKPOINT_PATH)
    print("Saved QNN checkpoint:", CHECKPOINT_PATH)

    if cfg.strict_attack_gate and float(qnn_asr["accuracy"]) < cfg.asr_min:
        raise RuntimeError(f"Blend ASR={float(qnn_asr['accuracy']):.4f} is below {cfg.asr_min:.2f}. Stop before detection.")

    build_and_save_pool_features(qnn, bundle, trigger, cfg, cfg.primary_seed)
    print("Notebook 1 complete. Upload/mount the full folder for Notebook 2:", cfg.out_dir)
    return attack_df

attack_success_df = run_notebook1()


100%|██████████| 19.7M/19.7M [00:01<00:00, 15.0MB/s]


Loaded dermamnist: train=torch.Size([7007, 3, 28, 28]), val=torch.Size([1003, 3, 28, 28]), test=torch.Size([2005, 3, 28, 28])


,class_id,class_name
0,0,actinic keratoses and intraepithelial carcinoma
1,1,basal cell carcinoma
2,2,benign keratosis-like lesions
3,3,dermatofibroma
4,4,melanoma
5,5,melanocytic nevi
6,6,vascular lesions


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 24.1MB/s]


...

,Attack,Dataset,Source,Target,Source→Target,Strength/Alpha,Poison Rate,CA,ASR,Gate Passed?,Status
0,blend,dermamnist,5,4,5->4,0.07,0.15,0.6925,1.0,True,passed_attack_gate


Saved QNN checkpoint: ./outputs_blend_dermamnist/blend_qnn_poisoned_checkpoint.pt
Building fixed 5% validation/test detection pools and saving reusable features...


,Group,Count
0,Clean source,450
1,Clean target,111
2,Poisonous triggered source,30
3,Total,591
4,Poison ratio,5.08%
5,Target poison ratio,5.00%


,Group,Count
0,Clean source,450
1,Clean target,223
2,Poisonous triggered source,35
3,Total,708
4,Poison ratio,4.94%
5,Target poison ratio,5.00%


Notebook 1 complete. Upload/mount the full folder for Notebook 2: ./outputs_blend_dermamnist


In [9]:

# ============================================================
# Q1 audit export: poison-rate feature pools, hard-negative pools, and split hashes
# ============================================================
# This cell does NOT retrain the poisoned QNN. It reloads the saved QNN checkpoint
# and exports leakage-safe feature tensors for 1%, 5%, 10%, and 15% detection pools.
# Notebook 2 consumes these saved artifacts for perfect-AUC audit, hard-negative tests,
# QNN-vs-classical comparison, and claim decision. No test-set tuning happens here.

RUN_POISON_RATE_SWEEP_FEATURE_EXPORT = True
RUN_HARD_NEGATIVE_FEATURE_EXPORT = True
AUDIT_POISON_RATES = [0.01, 0.05, 0.10]
AUDIT_SWEEP_DIR = os.path.join(cfg.out_dir, "poison_rate_sweep_features")
HARD_NEG_SWEEP_DIR = os.path.join(cfg.out_dir, "hard_negative_sweep_features")

import hashlib


def _hash_rows_np(arr, prefix=""):
    arr = np.asarray(arr)
    rows = []
    for i in range(len(arr)):
        h = hashlib.sha256(arr[i].tobytes()).hexdigest()
        rows.append({"row_id": i, "hash": prefix + h})
    return pd.DataFrame(rows)


def _save_rate_feature_tensor(rate_dir, name, arr):
    os.makedirs(rate_dir, exist_ok=True)
    torch.save(torch.tensor(np.asarray(arr), dtype=torch.float32), os.path.join(rate_dir, name))


def save_pool_metadata_rich(path, group, labels, class_labels=None, hard_negative=None, original_index=None):
    n = len(labels)
    df = pd.DataFrame({
        "row_id": np.arange(n, dtype=int),
        "group": np.asarray(group).astype(str),
        "poison_label": np.asarray(labels).astype(int),
    })
    if class_labels is not None:
        df["class_label"] = np.asarray(class_labels).astype(int)
    if hard_negative is not None:
        df["hard_negative"] = np.asarray(hard_negative).astype(bool)
    if original_index is not None:
        df["original_index"] = np.asarray(original_index).astype(int)
    df.to_csv(path, index=False)
    return df


def _save_detection_pool_feature_set(qnn, X_pool, y_poison, group, local_cfg, rate_dir, split_name, class_labels=None, hard_negative=None, original_index=None):
    q, qctx, _ = extract_qnn_features(qnn, X_pool, local_cfg, DEVICE)
    qprobe = extract_quantum_probe_features(qnn, X_pool, local_cfg, DEVICE)
    qstats = quantum_readout_statistics(q)
    raw = X_pool.reshape(len(X_pool), -1).numpy().astype(np.float32)

    suffix = f"_{split_name}.pt"
    _save_rate_feature_tensor(rate_dir, "features_qnn_context" + suffix, qctx)
    _save_rate_feature_tensor(rate_dir, "features_quantum_static" + suffix, q)
    _save_rate_feature_tensor(rate_dir, "features_quantum_probe_delta" + suffix, qprobe)
    _save_rate_feature_tensor(rate_dir, "features_quantum_readout_stats" + suffix, qstats)
    _save_rate_feature_tensor(rate_dir, "features_raw_pixels" + suffix, raw)
    torch.save(torch.tensor(y_poison, dtype=torch.long), os.path.join(rate_dir, f"labels_{split_name}.pt"))
    save_pool_metadata_rich(
        os.path.join(rate_dir, f"pool_metadata_{split_name}.csv"),
        group, y_poison, class_labels=class_labels, hard_negative=hard_negative, original_index=original_index,
    )
    _hash_rows_np(raw, prefix=f"{split_name}:").to_csv(os.path.join(rate_dir, f"raw_feature_hashes_{split_name}.csv"), index=False)
    return {
        "split": split_name,
        "total": int(len(y_poison)),
        "poison_count": int(np.asarray(y_poison).sum()),
        "poison_ratio_actual": float(np.asarray(y_poison).mean()),
        "clean_count": int(len(y_poison) - np.asarray(y_poison).sum()),
    }


def _choose(pool, n):
    n = int(max(0, min(n, len(pool))))
    return np.asarray(pool[:n], dtype=int)


def build_blend_hard_negative_detection_pool(X_raw, y, trigger, local_cfg, seed, poison_ratio=0.05):
    """
    Harder Blend detection pool for publication-level audit.

    Includes clean source, clean target, clean non-source/non-target, target-like clean samples,
    benign low-contrast hard negatives, and Blend-triggered source poisons.
    Uses only the provided split (validation or test), never test for validation decisions.
    """
    if not (0 < float(poison_ratio) < 0.5):
        raise ValueError(f"poison_ratio must be in (0, 0.5), got {poison_ratio}")
    rng = np.random.default_rng(seed)
    y_np = y.cpu().numpy().astype(int)
    src = rng.permutation(np.where(y_np == int(local_cfg.source_class))[0])
    tgt = rng.permutation(np.where(y_np == int(local_cfg.target_class))[0])
    oth = rng.permutation(np.where((y_np != int(local_cfg.source_class)) & (y_np != int(local_cfg.target_class)))[0])
    if len(src) < 20 or len(tgt) < 10 or len(oth) < 20:
        raise ValueError(f"Insufficient samples for hard pool: src={len(src)}, tgt={len(tgt)}, other={len(oth)}")

    # Counts are intentionally capped to keep Colab runtime manageable while making the task less trivial.
    n_clean_src = min(180, max(20, len(src) // 3))
    n_clean_tgt = min(120, max(20, len(tgt) // 2))
    n_target_like = min(80, max(0, len(tgt) - n_clean_tgt))
    n_clean_other = min(220, max(30, len(oth) // 3))
    n_hard_neg = min(160, max(20, len(oth) - n_clean_other))

    src_clean_idx = _choose(src, n_clean_src)
    src_poison_candidates = src[n_clean_src:]
    tgt_clean_idx = _choose(tgt, n_clean_tgt)
    tgt_like_idx = _choose(tgt[n_clean_tgt:], n_target_like)
    other_clean_idx = _choose(oth, n_clean_other)
    hard_base_idx = _choose(oth[n_clean_other:], n_hard_neg)

    X_source_clean = resize_tensor_images(X_raw[src_clean_idx], local_cfg.model_image_size)
    X_target_clean = resize_tensor_images(X_raw[tgt_clean_idx], local_cfg.model_image_size)
    X_target_like = resize_tensor_images(X_raw[tgt_like_idx], local_cfg.model_image_size) if len(tgt_like_idx) else torch.empty(0, 3, local_cfg.model_image_size, local_cfg.model_image_size)
    X_other_clean = resize_tensor_images(X_raw[other_clean_idx], local_cfg.model_image_size)
    X_hard_base = resize_tensor_images(X_raw[hard_base_idx], local_cfg.model_image_size)
    if len(X_hard_base):
        # Hard negative: benign low-contrast augmentation, not the attack trigger.
        X_hard = (0.97 * X_hard_base + 0.03 * benign_augment_for_blend(X_hard_base)).clamp(0, 1)
    else:
        X_hard = torch.empty(0, 3, local_cfg.model_image_size, local_cfg.model_image_size)

    clean_count = len(X_source_clean) + len(X_target_clean) + len(X_target_like) + len(X_other_clean) + len(X_hard)
    n_poison = int(round(clean_count * float(poison_ratio) / max(1e-8, 1.0 - float(poison_ratio))))
    n_poison = max(1, min(n_poison, len(src_poison_candidates)))
    poison_idx = _choose(src_poison_candidates, n_poison)
    X_poison_base = resize_tensor_images(X_raw[poison_idx], local_cfg.model_image_size)
    X_poison = apply_blend_trigger(X_poison_base, trigger, local_cfg.blend_alpha)

    X_parts = [X_source_clean, X_target_clean, X_other_clean, X_target_like, X_hard, X_poison]
    X_pool = torch.cat([p for p in X_parts if len(p) > 0], dim=0)

    groups, labels, class_labels, hard_flags, orig_idx = [], [], [], [], []
    def add(group, idxs, poison, hard=False):
        for ix in np.asarray(idxs, dtype=int):
            groups.append(group); labels.append(int(poison)); class_labels.append(int(y_np[ix])); hard_flags.append(bool(hard)); orig_idx.append(int(ix))
    add("clean_source", src_clean_idx, 0, False)
    add("clean_target", tgt_clean_idx, 0, False)
    add("clean_non_source_non_target", other_clean_idx, 0, False)
    add("target_like_clean", tgt_like_idx, 0, True)
    add("hard_negative_low_contrast_clean", hard_base_idx, 0, True)
    add("blend_triggered_source", poison_idx, 1, False)

    # Shuffle pool order so group order cannot be a detector shortcut.
    perm = rng.permutation(len(labels))
    X_pool = X_pool[perm]
    labels = np.asarray(labels, dtype=int)[perm]
    groups = np.asarray(groups, dtype=object)[perm]
    class_labels = np.asarray(class_labels, dtype=int)[perm]
    hard_flags = np.asarray(hard_flags, dtype=bool)[perm]
    orig_idx = np.asarray(orig_idx, dtype=int)[perm]
    return X_pool, labels, groups, class_labels, hard_flags, orig_idx


def export_feature_pools_from_checkpoint():
    if not RUN_POISON_RATE_SWEEP_FEATURE_EXPORT:
        print("RUN_POISON_RATE_SWEEP_FEATURE_EXPORT=False; skipping audit export.")
        return pd.DataFrame()
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Missing QNN checkpoint required for audit export: {CHECKPOINT_PATH}")
    os.makedirs(AUDIT_SWEEP_DIR, exist_ok=True)
    os.makedirs(HARD_NEG_SWEEP_DIR, exist_ok=True)

    bundle = load_dataset_bundle(cfg.dataset_name)

    # Split-level raw image hash audit. This verifies train/val/test disjointness independently of detector labels.
    split_hash_rows = []
    for split_name, X_split in [("train", bundle["X_train"]), ("val", bundle["X_val"]), ("test", bundle["X_test"])]:
        raw_split = X_split.reshape(len(X_split), -1).numpy().astype(np.float32)
        for row in _hash_rows_np(raw_split, prefix="").to_dict("records"):
            row["split"] = split_name
            split_hash_rows.append(row)
    split_hash_df = pd.DataFrame(split_hash_rows)[["split", "row_id", "hash"]]
    split_hash_df.to_csv(os.path.join(cfg.out_dir, "train_val_test_hash_audit.csv"), index=False)
    print("Saved split hash audit:", os.path.join(cfg.out_dir, "train_val_test_hash_audit.csv"))

    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    trigger = ckpt["trigger"].detach().cpu()
    qnn = QMedShieldHybridQNN(cfg).to(DEVICE)
    qnn.load_state_dict(ckpt["model_state_dict"])
    qnn.eval()

    manifest_rows = []
    for pool_type, root_dir in [("standard", AUDIT_SWEEP_DIR), ("hard_negative", HARD_NEG_SWEEP_DIR)]:
        if pool_type == "hard_negative" and not RUN_HARD_NEGATIVE_FEATURE_EXPORT:
            continue
        os.makedirs(root_dir, exist_ok=True)
        for rate in AUDIT_POISON_RATES:
            rate_tag = f"rate_{str(rate).replace('.', 'p')}"
            rate_dir = os.path.join(root_dir, rate_tag)
            os.makedirs(rate_dir, exist_ok=True)
            print(f"Exporting {pool_type} audit pool for poison_rate={rate:.2%} -> {rate_dir}")

            if pool_type == "standard":
                X_val_pool, y_val_poison, group_val, _ = build_blend_detection_pool(
                    bundle["X_val"], bundle["y_val"], trigger, cfg, cfg.primary_seed + 101, poison_ratio=float(rate)
                )
                X_test_pool, y_test_poison, group_test, _ = build_blend_detection_pool(
                    bundle["X_test"], bundle["y_test"], trigger, cfg, cfg.primary_seed + 202, poison_ratio=float(rate)
                )
                val_meta = _save_detection_pool_feature_set(qnn, X_val_pool, y_val_poison, group_val, cfg, rate_dir, "val")
                test_meta = _save_detection_pool_feature_set(qnn, X_test_pool, y_test_poison, group_test, cfg, rate_dir, "test")
            else:
                X_val_pool, y_val_poison, group_val, cls_val, hard_val, orig_val = build_blend_hard_negative_detection_pool(
                    bundle["X_val"], bundle["y_val"], trigger, cfg, cfg.primary_seed + 303, poison_ratio=float(rate)
                )
                X_test_pool, y_test_poison, group_test, cls_test, hard_test, orig_test = build_blend_hard_negative_detection_pool(
                    bundle["X_test"], bundle["y_test"], trigger, cfg, cfg.primary_seed + 404, poison_ratio=float(rate)
                )
                val_meta = _save_detection_pool_feature_set(qnn, X_val_pool, y_val_poison, group_val, cfg, rate_dir, "val", cls_val, hard_val, orig_val)
                test_meta = _save_detection_pool_feature_set(qnn, X_test_pool, y_test_poison, group_test, cfg, rate_dir, "test", cls_test, hard_test, orig_test)

            row = {
                "pool_type": pool_type,
                "requested_poison_rate": float(rate),
                "rate_tag": rate_tag,
                "rate_dir": rate_dir,
                "val_total": val_meta["total"],
                "val_poison_count": val_meta["poison_count"],
                "val_poison_ratio_actual": val_meta["poison_ratio_actual"],
                "test_total": test_meta["total"],
                "test_poison_count": test_meta["poison_count"],
                "test_poison_ratio_actual": test_meta["poison_ratio_actual"],
            }
            manifest_rows.append(row)
            with open(os.path.join(rate_dir, "rate_config.json"), "w") as f:
                json.dump(row, f, indent=2)

        manifest = pd.DataFrame([r for r in manifest_rows if r["pool_type"] == pool_type])
        manifest.to_csv(os.path.join(root_dir, f"{pool_type}_feature_manifest.csv"), index=False)

    all_manifest = pd.DataFrame(manifest_rows)
    all_manifest.to_csv(os.path.join(cfg.out_dir, "poison_rate_and_hard_negative_feature_manifest.csv"), index=False)
    print("Saved Q1 audit feature manifest:", os.path.join(cfg.out_dir, "poison_rate_and_hard_negative_feature_manifest.csv"))
    return all_manifest


# Call the optional trigger diagnostic only when explicitly enabled.
# Default path writes an audit artifact without loading data or training pilots.
try:
    if RUN_TRIGGER_TYPE_VALIDATION:
        trigger_diag_df = run_validation_only_trigger_type_test(load_dataset_bundle(cfg.dataset_name), cfg, cfg.primary_seed)
    else:
        trigger_diag_df = pd.DataFrame([{"Status": "disabled", "Reason": "RUN_TRIGGER_TYPE_VALIDATION=False; frozen config is used."}])
    trigger_diag_df.to_csv(os.path.join(cfg.out_dir, "trigger_type_validation_table.csv"), index=False)
except Exception as exc:
    pd.DataFrame([{"Status": "skipped_or_failed", "Reason": str(exc)}]).to_csv(os.path.join(cfg.out_dir, "trigger_type_validation_table.csv"), index=False)

q1_feature_manifest = export_feature_pools_from_checkpoint()
display(q1_feature_manifest)


Loaded dermamnist: train=torch.Size([7007, 3, 28, 28]), val=torch.Size([1003, 3, 28, 28]), test=torch.Size([2005, 3, 28, 28])


,class_id,class_name
0,0,actinic keratoses and intraepithelial carcinoma
1,1,basal cell carcinoma
2,2,benign keratosis-like lesions
3,3,dermatofibroma
4,4,melanoma
5,5,melanocytic nevi
6,6,vascular lesions


Saved split hash audit: ./outputs_blend_dermamnist/train_val_test_hash_audit.csv
Exporting standard audit pool for poison_rate=1.00% -> ./outputs_blend_dermamnist/poison_rate_sweep_features/rate_0p01


,Group,Count
0,Clean source,450
1,Clean target,111
2,Poisonous triggered source,6
3,Total,567
4,Poison ratio,1.06%
5,Target poison ratio,1.00%


,Group,Count
0,Clean source,450
1,Clean target,223
2,Poisonous triggered source,7
3,Total,680
4,Poison ratio,1.03%
5,Target poison ratio,1.00%


...

Exporting standard audit pool for poison_rate=10.00% -> ./outputs_blend_dermamnist/poison_rate_sweep_features/rate_0p1


,Group,Count
0,Clean source,450
1,Clean target,111
2,Poisonous triggered source,62
3,Total,623
4,Poison ratio,9.95%
5,Target poison ratio,10.00%


,Group,Count
0,Clean source,450
1,Clean target,223
2,Poisonous triggered source,75
3,Total,748
4,Poison ratio,10.03%
5,Target poison ratio,10.00%


Exporting hard_negative audit pool for poison_rate=1.00% -> ./outputs_blend_dermamnist/hard_negative_sweep_features/rate_0p01
Exporting hard_negative audit pool for poison_rate=5.00% -> ./outputs_blend_dermamnist/hard_negative_sweep_features/rate_0p05
Exporting hard_negative audit pool for poison_rate=10.00% -> ./outputs_blend_dermamnist/hard_negative_sweep_features/rate_0p1
Saved Q1 audit feature manifest: ./outputs_blend_dermamnist/poison_rate_and_hard_negative_feature_manifest.csv


,pool_type,requested_poison_rate,rate_tag,rate_dir,val_total,val_poison_count,val_poison_ratio_actual,test_total,test_poison_count,test_poison_ratio_actual
0,standard,0.01,rate_0p01,./outputs_blend_dermamnist/poison_rate_sweep_f...,567,6,0.010582,680,7,0.010294
1,standard,0.05,rate_0p05,./outputs_blend_dermamnist/poison_rate_sweep_f...,591,30,0.050761,708,35,0.049435
2,standard,0.10,rate_0p1,./outputs_blend_dermamnist/poison_rate_sweep_f...,623,62,0.099518,748,75,0.100267
3,hard_negative,0.01,rate_0p01,./outputs_blend_dermamnist/hard_negative_sweep...,517,5,0.009671,685,7,0.010219
4,hard_negative,0.05,rate_0p05,./outputs_blend_dermamnist/hard_negative_sweep...,539,27,0.050093,714,36,0.050420
5,hard_negative,0.10,rate_0p1,./outputs_blend_dermamnist/hard_negative_sweep...,569,57,0.100176,753,75,0.099602


In [10]:
# ============================================================
# Notebook 1 final export: ZIP outputs_blend_dermamnist/ for Notebook 2
# ============================================================
import os, zipfile


def zip_outputs_folder(out_dir=cfg.out_dir, zip_path=OUTPUT_ZIP_PATH):
    """Create a portable ZIP containing the full outputs_blend_dermamnist/ folder."""
    required_after_notebook1 = [
        "blend_qnn_poisoned_checkpoint.pt",
        "blend_config_used.json",
        "blend_attack_success.csv",
        "features_qnn_context_val.pt",
        "features_qnn_context_test.pt",
        "features_quantum_static_val.pt",
        "features_quantum_static_test.pt",
        "features_quantum_probe_delta_val.pt",
        "features_quantum_probe_delta_test.pt",
        "features_quantum_readout_stats_val.pt",
        "features_quantum_readout_stats_test.pt",
        "features_raw_pixels_val.pt",
        "features_raw_pixels_test.pt",
        "labels_val.pt",
        "labels_test.pt",
        "pool_metadata_val.csv",
        "pool_metadata_test.csv",
    ]
    missing = [fn for fn in required_after_notebook1 if not os.path.exists(os.path.join(out_dir, fn))]
    if missing:
        print("Cannot ZIP yet. Missing Notebook 1 outputs:")
        for fn in missing:
            print(" -", fn)
        raise FileNotFoundError("Notebook 1 did not finish all required outputs; ZIP export stopped.")

    optional_audit_files = [
        "train_val_test_hash_audit.csv",
        "poison_rate_and_hard_negative_feature_manifest.csv",
        "trigger_type_validation_table.csv",
    ]
    missing_optional = [fn for fn in optional_audit_files if not os.path.exists(os.path.join(out_dir, fn))]
    if missing_optional:
        print("Warning: optional Q1 audit outputs are missing; Notebook 2 will mark related checks as unresolved:")
        for fn in missing_optional:
            print(" -", fn)

    if os.path.exists(zip_path):
        os.remove(zip_path)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(out_dir):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=os.path.dirname(out_dir))
                zf.write(full_path, arcname)

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Created ZIP for Notebook 2: {zip_path} ({size_mb:.2f} MB)")
    return zip_path


zip_path = zip_outputs_folder()

if AUTO_ZIP_AND_DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files
        files.download(zip_path)
        print("Download started. Use this ZIP in Notebook 2:", os.path.basename(zip_path))
    except Exception as exc:
        print("ZIP created, but automatic Colab download did not start.")
        print("Manual download path:", zip_path)
        print("Reason:", exc)


Created ZIP for Notebook 2: /content/outputs_blend_dermamnist.zip (804.18 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Use this ZIP in Notebook 2: outputs_blend_dermamnist.zip
